# Portfolio Stress Lab

End-to-end walkthrough for loading a portfolio, running stress scenarios and reviewing the main risk diagnostics.

This notebook is designed to show both the analysis and the communication layer of the project.

## How to read this notebook

This notebook is written like a compact quant research memo:

- first load the portfolio and return history,
- then run the full stress engine,
- then interpret the base risk, scenario losses and attribution,
- and finally export a Markdown report for sharing.

It is meant to show both the analysis and the communication layer of the project.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'src' / 'stresslab').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root.')

ROOT = find_repo_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from stresslab import StressTestEngine
from stresslab.plotting import plot_scenario_losses

DATA = ROOT / 'data'
REPORTS = ROOT / 'reports'
REPORTS.mkdir(exist_ok=True)
ROOT

In [ ]:
returns = pd.read_csv(DATA / 'sample_returns.csv', index_col=0, parse_dates=True)
portfolio = pd.read_csv(DATA / 'sample_portfolio.csv')

engine = StressTestEngine(returns=returns, portfolio=portfolio, confidence_level=0.99)
report = engine.run_all(scenarios=ROOT / 'scenarios')

print(report.summary())

In [ ]:
display(Markdown('## Base risk metrics'))
display(report.scenario_results[['direct_loss', 'vol_corr_loss_proxy', 'total_loss']].sort_values('total_loss').style.format('{:.2%}'))
display(Markdown('## Top risk contributions'))
display(report.risk_contributions.head(10).to_frame('risk_contribution').style.format('{:.4f}'))
display(Markdown('## Sector exposure'))
display(report.sector_exposure.to_frame('weight').style.format('{:.2%}'))

In [ ]:
chart_path = REPORTS / 'notebook_scenario_losses.png'
plot_scenario_losses(report.scenario_results, chart_path)
ax = report.scenario_results['total_loss'].sort_values().plot(kind='barh', title='Scenario losses')
ax.set_xlabel('Estimated loss')
plt.tight_layout()
plt.show()
chart_path

## Next steps

- Replace the sample CSVs with your own holdings and return history.
- Add custom YAML scenarios under `scenarios/`.
- Export the final Markdown report for sharing.